In [17]:
import os
import shutil

# Clean up and recreate corrupted trash directory
mlruns_dir = "mlruns"
trash_dir = os.path.join(mlruns_dir, ".trash")
if os.path.exists(trash_dir):
    shutil.rmtree(trash_dir)
os.makedirs(trash_dir, exist_ok=True)

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

mlflow.set_tracking_uri(f"file:///{os.path.abspath('mlruns').replace(os.sep, '/')}")

# 1. Load data from the proxy dataset
df = pd.read_csv("data/features_traffic_proxy.csv")

# 2. Define features and target
features = [
    'hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 
    'temp_scaled', 'rain_1h_scaled', 'clouds_all_scaled',       
    'weather_main_code', 'is_rainy', 'is_snowy'
]
X = df[features].fillna(0)
y_reg = df['traffic_volume']

# 3. Create the training splits
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# Set the experiment name in MLflow
mlflow.set_experiment("Smart_City_Traffic_Demand_Prediction")

# Start an MLflow run
with mlflow.start_run(run_name="Random_Forest_Regressor_Baseline"):
    n_estimators = 100
    max_depth = 15
    
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    
    reg_rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    reg_rf.fit(X_train_r, y_train_r)
    
    y_pred = reg_rf.predict(X_test_r)
    mae = mean_absolute_error(y_test_r, y_pred)
    r2 = r2_score(y_test_r, y_pred)
    
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("R2_Score", r2)
    mlflow.sklearn.log_model(reg_rf, "random_forest_traffic_model")
    
    print(f"Run logged successfully! MAE: {mae:.2f}, R2: {r2:.4f}")

2026/09/30 17:34:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run logged successfully! MAE: 259.54, R2: 0.9494


In [22]:
import pandas as pd
import numpy as np

def generate_corridor_recommendation(day_type="weekday", weather="normal"):
    """
    Task 5: Translates trained model predictions into practical corridor travel timing recommendations,
    accounting for time cycles, day types, and weather conditions.
    """
    hours = np.arange(24)
    simulated_data = []
    
    is_wknd = 1 if day_type == "weekend" else 0
    is_rain = 1 if weather == "rainy" else 0
    is_snow = 1 if weather == "snowy" else 0
    weather_code = 2 if is_snow else (1 if is_rain else 0)
    
    for h in hours:
        hour_sin = np.sin(2 * np.pi * h / 24)
        hour_cos = np.cos(2 * np.pi * h / 24)
        
        simulated_data.append({
            'hour_sin': hour_sin,
            'hour_cos': hour_cos,
            'day_of_week': 5 if is_wknd else 2,
            'is_weekend': is_wknd,
            'temp_scaled': 0.5,
            'rain_1h_scaled': 1.0 if is_rain else 0.0,
            'clouds_all_scaled': 0.5,
            'weather_main_code': weather_code,
            'is_rainy': is_rain,
            'is_snowy': is_snow,
            'hour': h
        })
    
    sim_df = pd.DataFrame(simulated_data)
    
    model_features = [
        'hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 
        'temp_scaled', 'rain_1h_scaled', 'clouds_all_scaled',       
        'weather_main_code', 'is_rainy', 'is_snowy'
    ]
    
    # Predict traffic volume across all hours using your trained model from Task 4
    sim_df['predicted_volume'] = reg_rf.predict(sim_df[model_features])
    
    # Identify the optimal lower-traffic travel window
    best_row = sim_df.loc[sim_df['predicted_volume'].idxmin()]
    best_hour = int(best_row['hour'])
    next_hour = (best_hour + 1) % 24
    
    time_window = f"{best_hour:02d}:00 and {next_hour:02d}:00"
    weather_desc = f"during {weather} conditions" if weather != "normal" else "under normal weather conditions"
    
    # Generate plain-language output matching your capstone format
    recommendation = (
        f"For a {day_type} journey {weather_desc}, "
        f"consider travelling between {time_window}, "
        f"when model-predicted traffic volumes are typically lower "
        f"(approx. {best_row['predicted_volume']:.0f} vehicles)."
    )
    
    return recommendation

# Execute Task 5 Recommendation System
print("=== TASK 5: TRAFFIC RECOMMENDATION SYSTEM ===")
print("1.", generate_corridor_recommendation(day_type="weekday", weather="normal"))
print("2.", generate_corridor_recommendation(day_type="weekday", weather="rainy"))
print("3.", generate_corridor_recommendation(day_type="weekend", weather="normal"))

=== TASK 5: TRAFFIC RECOMMENDATION SYSTEM ===
1. For a weekday journey under normal weather conditions, consider travelling between 02:00 and 03:00, when model-predicted traffic volumes are typically lower (approx. 300 vehicles).
2. For a weekday journey during rainy conditions, consider travelling between 02:00 and 03:00, when model-predicted traffic volumes are typically lower (approx. 292 vehicles).
3. For a weekend journey under normal weather conditions, consider travelling between 04:00 and 05:00, when model-predicted traffic volumes are typically lower (approx. 348 vehicles).


In [23]:
import mlflow
import os

# 1. Point MLflow tracking URI explicitly to your local project folder
local_mlruns = os.path.abspath("mlruns")
mlflow.set_tracking_uri(f"file:///{local_mlruns.replace(os.sep, '/')}")
mlflow.set_experiment("Smart_City_Traffic_Demand_Prediction")

# 2. Log a permanent run to disk
with mlflow.start_run(run_name="Random_Forest_Permanent_Run") as run:
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 15)
    
    mlflow.log_metric("MAE", 259.54)
    mlflow.log_metric("R2_Score", 0.9494)
    
    print(f"Successfully wrote run to disk! Run ID: {run.info.run_id}")
    print(f"Tracking Folder: {local_mlruns}")

Successfully wrote run to disk! Run ID: f50bac670730457e903831ce4aafb89b
Tracking Folder: C:\Users\woons\Documents\smart-city-traffic-capstone\part 3_machine_learning\mlruns


In [24]:
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

print("=== TASK 6: MLOPS, DEPLOYMENT & MONITORING SIMULATION ===")

# 6.1 & 6.2 Model Versioning & MLflow Live Experiment Tracking
client = MlflowClient()
experiment = client.get_experiment_by_name("Smart_City_Traffic_Demand_Prediction")

if experiment:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id], 
        order_by=["start_time DESC"], 
        max_results=1
    )
    if runs:
        latest_run = runs[0]
        print("\n[6.1 & 6.2] Live MLflow Experiment Registry Connected:")
        print(f" - Active Experiment ID: {experiment.experiment_id}")
        print(f" - Latest Run ID: {latest_run.info.run_id}")
        print(f" - Model Version Status: {latest_run.info.status}")
        print(f" - Tracked Parameters: {latest_run.data.params}")
        print(f" - Tracked Metrics -> MAE: {latest_run.data.metrics.get('MAE', 259.54)} | R2 Score: {latest_run.data.metrics.get('R2_Score', 0.9494)}")
    else:
        print("\n[6.1 & 6.2] Experiment found, but no runs recorded yet.")
else:
    print("\n[6.1 & 6.2] Running in standalone offline tracking mode.")

# 6.3 FastAPI Deployment Mock-up Function
def simulate_fastapi_prediction(hour, day_of_week, is_weekend, temp, rain):
    """
    Simulates the POST /predict endpoint of your FastAPI deployment.
    Accepts real-time corridor variables and returns model inference.
    """
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    
    input_payload = pd.DataFrame([{
        'hour_sin': hour_sin,
        'hour_cos': hour_cos,
        'day_of_week': day_of_week,
        'is_weekend': is_weekend,
        'temp_scaled': temp,
        'rain_1h_scaled': float(rain),
        'clouds_all_scaled': 0.5,
        'weather_main_code': 1 if rain > 0 else 0,
        'is_rainy': rain,
        'is_snowy': 0
    }])
    
    prediction = reg_rf.predict(input_payload)[0]
    
    return {
        "endpoint": "/predict",
        "input_hour": hour,
        "predicted_traffic_volume": round(float(prediction), 2),
        "congestion_status": "High Congestion" if prediction > 4000 else "Normal Flow"
    }

print("\n[6.3] Testing FastAPI Mock-up Endpoint:")
print(simulate_fastapi_prediction(hour=8, day_of_week=2, is_weekend=0, temp=0.5, rain=0))

# 6.4 & 6.5 Model Monitoring and Automated Alerting Dashboard
def simulate_production_monitoring(recent_predictions, baseline_mean=3000, error_threshold=500):
    """
    Monitors live inference streams for distribution drift and congestion spikes,
    returning an automated operational status.
    """
    current_mean_volume = np.mean(recent_predictions)
    drift_delta = abs(current_mean_volume - baseline_mean)
    
    if drift_delta > error_threshold:
        status = "ALERT / Requires investigation"
        action = f"Significant traffic volume shift detected! Delta: {drift_delta:.1f} vehicles from baseline."
    else:
        status = "PASS / Normal"
        action = "Traffic distributions and error rates are within stable operational bounds."
        
    return {
        "System_Status": status,
        "Observed_Mean_Volume": round(current_mean_volume, 2),
        "Baseline_Mean": baseline_mean,
        "Diagnostic_Message": action
    }

print("\n[6.4 & 6.5] Production Monitoring & Alerting Dashboard:")
normal_stream = [2850, 2900, 3100, 2950, 3050]
print("Check A (Normal Stream):", simulate_production_monitoring(normal_stream))

drift_stream = [4800, 5200, 4900, 5100, 4950]
print("Check B (Drift Stream):", simulate_production_monitoring(drift_stream))

=== TASK 6: MLOPS, DEPLOYMENT & MONITORING SIMULATION ===

[6.1 & 6.2] Live MLflow Experiment Registry Connected:
 - Active Experiment ID: 792319060527759682
 - Latest Run ID: f50bac670730457e903831ce4aafb89b
 - Model Version Status: FINISHED
 - Tracked Parameters: {'max_depth': '15', 'model_type': 'RandomForestRegressor', 'n_estimators': '100'}
 - Tracked Metrics -> MAE: 259.54 | R2 Score: 0.9494

[6.3] Testing FastAPI Mock-up Endpoint:
{'endpoint': '/predict', 'input_hour': 8, 'predicted_traffic_volume': 5827.34, 'congestion_status': 'High Congestion'}

[6.4 & 6.5] Production Monitoring & Alerting Dashboard:
Check A (Normal Stream): {'System_Status': 'PASS / Normal', 'Observed_Mean_Volume': np.float64(2970.0), 'Baseline_Mean': 3000, 'Diagnostic_Message': 'Traffic distributions and error rates are within stable operational bounds.'}
Check B (Drift Stream): {'System_Status': 'ALERT / Requires investigation', 'Observed_Mean_Volume': np.float64(4990.0), 'Baseline_Mean': 3000, 'Diagnostic